In [19]:
import os
import pickle
import numpy as np
import pandas as pd
def compute_cluster_activation_stats(base_directory=["../"]):
    """
    Reads ActivationRanges.pkl files from subdirectories (pruning %)
    and computes avg min, avg max, and range per cluster.

    Returns:
    --------
    dict:
        {
          model_name: {
            pruning_pct: {
              'c1': {'avg_min': ..., 'avg_max': ..., 'range': ...},
              'c2': {...},
              'c3': {...}
            }
          }
        }
    """
    cluster_stats = {}
    runs = 0
    for base_dir in base_directory:
        for iteration, percent in enumerate(os.listdir(base_dir)):
            pkl_path = os.path.join(base_dir, percent, "ActivationRanges.pkl")
            if not os.path.isfile(pkl_path):
                continue

            with open(pkl_path, "rb") as f:
                res = pickle.load(f)

            # Initialize accumulators
            mins = {"c1": 0.0, "c2": 0.0, "c3": 0.0}
            maxs = {"c1": 0.0, "c2": 0.0, "c3": 0.0}
            counts = {"c1": 0, "c2": 0, "c3": 0}

            for neuron in res:
                for idx, cname in enumerate(["c1", "c2", "c3"]):
                    try:
                        mins[cname] += neuron[idx][0]
                        maxs[cname] += neuron[idx][1]
                        counts[cname] += 1
                    except Exception:
                        continue

            # Build stats for this pruning percentage
            pct_stats = {}
            for cname in ["c1", "c2", "c3"]:
                if counts[cname] == 0:
                    continue
                avg_min = mins[cname] / counts[cname]
                avg_max = maxs[cname] / counts[cname]
                if cname in pct_stats:
                    runs += 1
                    pct_stats[cname]['avg_min'] += avg_min
                    pct_stats[cname]['avg_max'] += avg_max
                    pct_stats[cname]['range'] += avg_max - avg_min
                else:
                    runs += 1
                    pct_stats[cname] = {
                        "avg_min": avg_min,
                        "avg_max": avg_max,
                        "range": avg_max - avg_min
                    }
            # Store (using a single model key, adjust if needed)
            model = "bert"   # or whatever model name you want
            cluster_stats.setdefault(model, {})[iteration] = pct_stats
        print(pct_stats)
        for cname in pct_stats:
            for key in pct_stats[cname]:
                pct_stats[cname][key] /= runs
        

    return cluster_stats

def create_activation_tables(cluster_stats, output_prefix='activation_table'):
    """
    Create tables with clusters on rows and pruning percentages on columns
    for both average min and average max activations.
    
    Parameters:
    -----------
    cluster_stats : dict
        Nested dictionary from compute_cluster_statistics function
    output_prefix : str
        Prefix for output CSV files
    
    Returns:
    --------
    dict : Dictionary of DataFrames for each model and metric
    """
    
    tables = {}
    
    for model in cluster_stats:
        data = cluster_stats[model]
        pruning_pcts = sorted(data.keys())
        clusters = sorted(list(set([c for pct in data for c in data[pct]])))
        
        # Create table for average min activations
        min_data = []
        for cluster in clusters:
            row = [data[idx][cluster]['avg_min'] if cluster in data[idx] else np.nan 
                   for idx, pct in enumerate(pruning_pcts)]
            min_data.append(row)
        
        df_min = pd.DataFrame(
            min_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_min.index.name = 'Cluster'
        
        # Create table for average max activations
        max_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['avg_max'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            max_data.append(row)
        
        df_max = pd.DataFrame(
            max_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_max.index.name = 'Cluster'
        
        range_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['range'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            range_data.append(row)
        
        df_range = pd.DataFrame(
            range_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_range.index.name = 'Cluster'
        
        # Store tables
        tables[f'{model}_avg_min'] = df_min
        tables[f'{model}_avg_max'] = df_max
        tables[f'{model}_range'] = df_range
        
        # Save to CSV
        
        print(f"\n{model} - Average Min Activations:")
        print(df_min.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Average Max Activations:")
        print(df_max.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Range Activations:")
        print(df_range.to_string(float_format='%.4f'))
    
    return tables


In [20]:
cluster_stats = compute_cluster_activation_stats(["/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Masks", "/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_6/Masks"])
tables = create_activation_tables(cluster_stats)


{'c1': {'avg_min': 0.00019190784274746784, 'avg_max': 0.5327609220601948, 'range': 0.5325690142174473}, 'c2': {'avg_min': 0.5333087081351433, 'avg_max': 1.1855486329181126, 'range': 0.6522399247829693}, 'c3': {'avg_min': 1.1869475385510777, 'avg_max': 3.083429785497696, 'range': 1.8964822469466183}}
{'c1': {'avg_min': 0.00018123798057284398, 'avg_max': 0.4952589164226513, 'range': 0.49507767844207845}, 'c2': {'avg_min': 0.49577730609278375, 'avg_max': 1.1029764605860404, 'range': 0.6071991544932567}, 'c3': {'avg_min': 1.1043252792851679, 'avg_max': 2.8807950347515163, 'range': 1.7764697554663484}}

bert - Average Min Activations:
            0%     1%     2%     3%     4%     5%
Cluster                                          
c1      0.0002 0.0002 0.0002 0.0002 0.0002 0.0000
c2      0.5298 0.4891 0.5036 0.5022 0.4981 0.0138
c3      1.1687 1.0907 1.1210 1.1179 1.1110 0.0307

bert - Average Max Activations:
            0%     1%     2%     3%     4%     5%
Cluster                      